## Import dependencies

In [ ]:
import json
import gzip
import pandas as pd
import requests

## working with downloaded data

In [ ]:
with gzip.open("../../data/meta_Electronics.jsonl.gz", "rt") as f:
    first_line = json.loads(f.readline())

In [ ]:
first_line

## filter out only recent items (appeared in stock > 2022)

In [ ]:
def filter_data_by_date(data: dict) -> bool:
    filter = False
    if int(data["details"]["Date First Available"][-4:]) > 2022:
        filter = True

    return filter

In [ ]:
test_dict = {'main_category': 'Gift Cards',
 'title': 'Amazon.com Gift Card in Gift Tag (Various Designs)',
 'average_rating': 4.8,
 'rating_number': 1006,
 'features': ['Gift Card is affixed inside a gift tag',
  'Gift amount may not be printed on Gift Cards',
  'Gift Card has no fees and no expiration date',
  'No returns and no refunds on Gift Cards',
  'Gift Card is redeemable towards millions of items storewide at Amazon.com',
  'Scan and redeem any Gift Card with a mobile or tablet device via the Amazon App',
  'Free One-Day Shipping (where available)',
  'Customized gift message, if chosen at check-out, only appears on packing slip and not on the actual gift card or carrier'],
 'description': ["Amazon.com Gift Cards are the perfect way to give them exactly what they're hoping for - even if you don't know what it is. Amazon.com Gift Cards are redeemable for millions of items across Amazon.com. Item delivered is a single physical Amazon.com Gift Card nested inside or with a free gift accessory."],
 'price': None,
 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41ZA96xtATL._SX38_SY50_CR,0,0,38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41ZA96xtATL.jpg',
   'variant': 'MAIN',
   'hi_res': 'https://m.media-amazon.com/images/I/71cWJvVGYtL._SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/41NK1FX6uUL._SX38_SY50_CR,0,0,38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41NK1FX6uUL.jpg',
   'variant': 'PT01',
   'hi_res': 'https://m.media-amazon.com/images/I/71q-qp4b3-L._SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/41Y45S0GirL._SX38_SY50_CR,0,0,38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41Y45S0GirL.jpg',
   'variant': 'PT02',
   'hi_res': 'https://m.media-amazon.com/images/I/71KutAnl9gL._SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/417MZ16DhcL._SX38_SY50_CR,0,0,38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/417MZ16DhcL.jpg',
   'variant': 'PT03',
   'hi_res': 'https://m.media-amazon.com/images/I/61FMUKaXfJL._SL1175_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/21-tRQuOBZL._SX38_SY50_CR,0,0,38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/21-tRQuOBZL.jpg',
   'variant': 'PT12',
   'hi_res': 'https://m.media-amazon.com/images/I/61blLcj3pWL._SL1500_.jpg'}],
 'videos': [],
 'store': 'Amazon',
 'categories': ['Gift Cards', 'Gift Card Recipients', 'For Him'],
 'details': {'Package Dimensions': '5 x 3 x 0.1 inches; 0.63 Ounces',
  'Item model number': 'Fixed',
  'Date First Available': 'August 29, 2017',
  'Manufacturer': 'Amazon'},
 'parent_asin': 'B06ZXTKYHN',
 'bought_together': None}

In [ ]:
filter_data_by_date(test_dict)

In [ ]:
def process_electronics_data():
    input_path = "../../data/meta_Electronics.jsonl.gz"
    matched_path = "../../data/2022-2023.jsonl"
    no_date_path = "../../data/no_date.jsonl"

    processed = 0
    with gzip.open(input_path, "rt") as infile, \
         open(matched_path, "w") as matched_file, \
         open(no_date_path, "w") as no_date_file:
        for line in infile:
            try:
                record = json.loads(line)
                if filter_data_by_date(record):
                    matched_file.write(line)
            except Exception:
                no_date_file.write(line)

            processed += 1
            if processed % 10_000 == 0:
                print(f"Processed {processed} records")

    print(f"Finished processing {processed} records")


process_electronics_data()

## Explore distribution catgory

In [ ]:
df = pd.read_json("../../data/2022-2023.jsonl", lines=True)

In [ ]:
df.head()

In [ ]:
df["main_category"].value_counts().plot(kind="bar")

In [ ]:
df_ratings_100 = df[df["rating_number"] >= 100]

In [ ]:
len(df)

In [ ]:
len(df_ratings_100)

In [ ]:
df_ratings_100["main_category"].value_counts().plot(kind="bar")

## explore distribution ratings

In [ ]:
df_ratings_100["average_rating"].plot(kind="hist", bins=50, range=(0,5))

### sample 6

In [ ]:
df_sample_1000 = df_ratings_100.sample(n=1000, random_state=42)

In [ ]:
len(df_sample_1000)

In [ ]:
df_sample_1000["average_rating"].plot(kind="hist", bins=50, range=(0,5))

In [ ]:
df_sample_1000["price"].plot(kind="hist", bins=100, range=(0,500))

In [ ]:
df_sample_1000["main_category"].value_counts().plot(kind="bar")

In [ ]:
df_ratings_100.to_json("../../data/meta_electronics_2022_2023_with_category_ratings_100.jsonl", orient='records', lines=True)
df_sample_1000.to_json("../../data/meta_electronics_2022_2023_with_category_ratings_100_samples_1000.jsonl", orient='records', lines=True)

### Extract ratings that match sample data

In [ ]:
df_ratings_100 = pd.read_json("../../data/meta_electronics_2022_2023_with_category_ratings_100.jsonl", lines=True)
df_sample_1000 = pd.read_json("../../data/meta_electronics_2022_2023_with_category_ratings_100_samples_1000.jsonl", lines=True)

In [ ]:
with gzip.open("../../data/Electronics.jsonl.1.gz", "r") as fp:
    with open("../../data/Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", 'a') as fp_out:
        id_list = set(df_sample_1000['parent_asin'].values)
        i =0
        for line in fp:
            data = json.loads(line.strip())
            if data['parent_asin'] in id_list:
                json.dump(data, fp_out)
                fp_out.write('\n')
                fp_out.flush()
            i += 1
            if i % 100000 == 0:
                print(f"processed {i} lines")